# Week 3: Not All Lost Customers Cost the Same
## Latent Class Analysis (GMM) with CLV-Weighted Prioritization

**Masterclass:** [Week 3 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week3_churn_segmentation.html)

**Dataset:** IBM Telco Customer Churn (same as Week 2)

**What You'll Build:**
1. Feature engineering for churner profiles
2. Gaussian Mixture Model as LCA proxy
3. Optimal cluster selection via BIC
4. CLV-weighted segment prioritization
5. Segment-specific retention strategies

In [ ]:
!pip install lifelines pandas matplotlib seaborn scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Prepare Churner Profiles

We focus exclusively on churned customers and build behavioral profiles for segmentation.

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['churned'] = (df['Churn'] == 'Yes').astype(int)

# Focus on churners only
churners = df[df['churned']==1].copy()
print(f"Churned customers: {len(churners):,}")

# Build features for segmentation
churners['fiber_optic'] = (churners['InternetService']=='Fiber optic').astype(int)
churners['month_to_month'] = (churners['Contract']=='Month-to-month').astype(int)
churners['electronic_check'] = (churners['PaymentMethod']=='Electronic check').astype(int)
churners['has_tech_support'] = (churners['TechSupport']=='Yes').astype(int)
churners['has_online_security'] = (churners['OnlineSecurity']=='Yes').astype(int)
churners['has_streaming'] = ((churners['StreamingTV']=='Yes') | (churners['StreamingMovies']=='Yes')).astype(int)

features = ['tenure', 'MonthlyCharges', 'TotalCharges',
            'fiber_optic', 'month_to_month', 'electronic_check',
            'has_tech_support', 'has_online_security', 'has_streaming']

X = churners[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features: {features}")
print(f"\nFeature means for churners:")
print(X.mean().round(2).to_string())

---
## Part 2: Optimal Cluster Count (BIC Method)

We test 2-8 clusters and pick the one with the lowest BIC (Bayesian Information Criterion). BIC penalizes model complexity, preventing overfitting.

In [ ]:
bic_scores = {}
for k in range(2, 9):
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=5)
    gmm.fit(X_scaled)
    bic_scores[k] = gmm.bic(X_scaled)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(bic_scores.keys()), list(bic_scores.values()),
        'o-', color='#264653', linewidth=2, markersize=8)
best_k = min(bic_scores, key=bic_scores.get)
ax.axvline(x=best_k, color='#c45d3e', linestyle='--', label=f'Best: k={best_k}')
ax.set_xlabel('Number of Clusters')
ax.set_ylabel('BIC (lower is better)')
ax.set_title('Optimal Cluster Count', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Optimal clusters: {best_k}")

---
## Part 3: Fit Final Model & Profile Segments

In [ ]:
gmm = GaussianMixture(n_components=best_k, random_state=42, n_init=10)
churners['segment'] = gmm.fit_predict(X_scaled)

print("=== Segment Profiles ===\n")
for seg in sorted(churners['segment'].unique()):
    s = churners[churners['segment']==seg]
    print(f"--- Segment {seg} ({len(s)} customers, {len(s)/len(churners):.1%}) ---")
    print(f"  Avg tenure:          {s['tenure'].mean():.0f} months")
    print(f"  Avg monthly charges: ${s['MonthlyCharges'].mean():.0f}")
    print(f"  Avg total charges:   ${s['TotalCharges'].mean():,.0f}")
    print(f"  Month-to-month:      {s['month_to_month'].mean():.0%}")
    print(f"  Fiber optic:         {s['fiber_optic'].mean():.0%}")
    print(f"  Electronic check:    {s['electronic_check'].mean():.0%}")
    print(f"  Has tech support:    {s['has_tech_support'].mean():.0%}")
    print(f"  Has streaming:       {s['has_streaming'].mean():.0%}")
    print()

In [ ]:
# Heatmap of segment profiles (normalized)
profile = churners.groupby('segment')[features].mean()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(profile_norm.T, annot=profile.T.round(1), fmt='', cmap='YlOrRd',
            ax=ax, cbar_kws={'label': 'Normalized Value'})
ax.set_title('Segment Profiles (Heatmap)', fontsize=14, fontweight='bold')
ax.set_xlabel('Segment')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

---
## Part 4: CLV-Weighted Prioritization

Not all segments deserve equal retention investment. We weight by CLV at stake.

In [ ]:
MARGIN = 0.40
churners['approx_clv'] = churners['TotalCharges'] * MARGIN

clv_summary = churners.groupby('segment').agg(
    count=('approx_clv', 'size'),
    avg_clv=('approx_clv', 'mean'),
    total_clv=('approx_clv', 'sum')
).round(0)
clv_summary['pct_of_total'] = (clv_summary['total_clv']/clv_summary['total_clv'].sum()*100).round(1)
clv_summary = clv_summary.sort_values('total_clv', ascending=False)

print("=== CLV at Stake by Segment ===")
print(clv_summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
clv_summary['total_clv'].plot(kind='bar', ax=axes[0], color='#264653')
axes[0].set_title('Total CLV at Stake')
axes[0].set_ylabel('Total CLV ($)')

clv_summary['avg_clv'].plot(kind='bar', ax=axes[1], color='#c45d3e')
axes[1].set_title('Avg CLV per Customer')
axes[1].set_ylabel('Avg CLV ($)')
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Name Your Segments
Give each segment a business-friendly name (e.g., "Trial-and-Bail", "Poached Whale", "Price-Sensitive Loyalist"). Justify with data.

### Exercise 2: Retention Strategy Matrix
For each segment, propose a strategy: discount? better onboarding? tech support? account manager? Set budget as % of CLV at stake.

### Exercise 3: Predict Segments for Active Customers
Use `gmm.predict()` on active (non-churned) customers to proactively identify which segment they'd fall into if they churned.

```python
active = df[df['churned']==0].copy()
# ... create same features ...
active['predicted_churn_segment'] = gmm.predict(scaler.transform(active[features]))
```

---
## Key Takeaways

1. **Not all churn is the same** — segmentation reveals distinct failure patterns
2. **BIC** prevents over/under-segmenting
3. **CLV weighting** directs budget to where the dollars are
4. **Proactive segmentation** of active customers enables preemptive intervention

**Next:** [Week 4 — Complete CLV Engine](https://balaviswanath-analytics.github.io/analytics_masterclass/week4_recency_frequency.html)